# Evaluating anomaly detectors

This notebook does the same as `evaluate.py`. It answers the question: **if the trigger may keep only a small
fraction of normal (zero-bias) events, how much of each signal does each model catch?**

<!-- TODO(organisers): explain trigger acceptance / rate and why we fix it. -->

In [ ]:
from pathlib import Path

import torch

from dataloader import create_dataloaders
from models import registry
from models.sklearn_wrapper import SklearnModelWrapper
from utils import compute_scores, efficiency_at_acceptance, get_device, roc
import plotting

device = get_device("auto")

In [ ]:
# (display name, path to a saved model)
MODELS = [
    ("dense-ae", "output/dense-ae/best_model.pth"),
]
CONFIG = "config/baseline.yml"

# Fraction of zero-bias events the trigger may keep.
# TODO(organisers): choose the operating point for the hackathon.
ACCEPTANCE = 1e-3


def load_model(path):
    if str(path).endswith(".pkl"):
        return SklearnModelWrapper.load(path)
    return torch.load(path, map_location=device, weights_only=False).to(device)

In [ ]:
test_loaders = {}  # one set of test loaders per input type
scores = {}        # scores[model][sample] = anomaly scores

for model_name, model_path in MODELS:
    model = load_model(model_path)
    architecture, model_cfg = registry.class_to_config(type(model))
    input_type = model_cfg["input_type"]
    if input_type not in test_loaders:
        _, _, test_loaders[input_type] = create_dataloaders(
            CONFIG, shuffle=False, input_type=input_type, load_test=True
        )
    scores[model_name] = {
        name: compute_scores(model, loader, device) for name, loader in test_loaders[input_type].items()
    }

In [ ]:
efficiencies = {}
for model_name, s in scores.items():
    rocs, efficiencies[model_name] = {}, {}
    for signal in s:
        if signal == "background":
            continue
        rocs[signal] = roc(s["background"], s[signal])
        efficiencies[model_name][signal] = efficiency_at_acceptance(s["background"], s[signal], ACCEPTANCE)
        print(f"{model_name:15s} {signal:35s} AUC = {rocs[signal][2]:.3f}   "
              f"efficiency @ {ACCEPTANCE:g} = {efficiencies[model_name][signal]:.4f}")

    plotting.plot_score_distributions(s, title=model_name)
    plotting.plot_roc(rocs, ACCEPTANCE, title=model_name)

In [ ]:
plotting.plot_signal_efficiency(efficiencies, ACCEPTANCE);

## Discussion

<!-- TODO(organisers): questions. Which signals are easy or hard to find, and why? How does the model compare to the existing trigger (L1bit)? -->